# Phase 5: Offline Evaluation of Causal-RL Agent (MIND-large)

Evaluates the trained PPO agent from Phase 4 on held-out test sessions from MIND-large.
Compares against Random and Popularity baselines using NDCG@K, Precision@K, and ILD.

## Tuning for MIND-large
- More test sessions available for statistically robust evaluation.
- Significance thresholds remain at p < 0.01.
- Baseline methods (Random, Popularity) are computed on the MIND-large test set.

## Inputs
- `data/scm_test.parquet` — held-out test impressions from MIND-large
- `artifacts/checkpoints/ppo_causal_rs_large_w03.zip` — trained PPO policy

## Outputs
- Per-metric comparison table
- Significance test results

In [ ]:
import warnings
import logging
import pickle
import sys
import time
from pathlib import Path
from collections import Counter
import ast

import numpy as np
import pandas as pd
from scipy import stats
import torch

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.evaluation.metrics import (
    ndcg_at_k, precision_at_k, ild,
)
from src.rl_agent.environment import NewsRecommendEnv

from stable_baselines3 import PPO

warnings.filterwarnings('ignore')
logging.getLogger('src').setLevel(logging.WARNING)

DATA = _root / "data"
ARTIFACTS = _root / "artifacts"
CHECKPOINTS = ARTIFACTS / "checkpoints"
print(f"Project root: {_root}")
print(f"Dataset: MIND-large")
print(f"Checkpoints: {CHECKPOINTS}")

## 2. Load Test Data
Load the MIND-large test parquet and the CDI cache.

In [ ]:
test_path = DATA / "scm_test.parquet"
try:
    test_df = pd.read_parquet(test_path)
    print(f"Loaded MIND-large test data: {len(test_df)} rows, {test_df['user_id'].nunique()} users, {test_df['impression_id'].nunique()} impressions")
except FileNotFoundError:
    print(f"ERROR: {test_path} not found. Run Phase 1 first.")
    raise

cdi_path = ARTIFACTS / "cdi_cache.pkl"
try:
    with open(cdi_path, "rb") as f:
        cdi_cache = pickle.load(f)
    print(f"Loaded CDI cache: {len(cdi_cache)} entries")
except FileNotFoundError:
    print(f"CDI cache not found at {cdi_path}; RL evaluation will use zero CDI")
    cdi_cache = {}


## 3. Build Test Sessions

In [ ]:
def _to_array(val):
    if isinstance(val, (list, np.ndarray)):
        return np.array(val, dtype=np.float32)
    if isinstance(val, str):
        return np.array(ast.literal_eval(val), dtype=np.float32)
    raise TypeError(f"Unexpected embedding type {type(val)}")


test_sessions = []
for imp_id, group in test_df.groupby("impression_id", sort=False):
    user_id = group["user_id"].iloc[0]
    history_emb = _to_array(group["U_history_emb_full"].iloc[0])
    item_ids = group["item_id"].tolist()
    clicks = group["Y_click"].tolist()
    title_embs = group["I_title_emb_full"].apply(_to_array).tolist()
    candidates = []
    for iid, emb in zip(item_ids, title_embs):
        candidates.append(type("C", (), {"item_id": iid, "title_emb": emb})())
    session = type("Session", (), {
        "user_id": user_id,
        "initial_history_emb": history_emb,
        "candidates": [candidates],
        "clicks": [clicks],
        "clicked_items": [item_ids[i] for i, c in enumerate(clicks) if c == 1],
    })()
    test_sessions.append(session)

print(f"Built {len(test_sessions)} test sessions from MIND-large")


## 4. Evaluate PPO (Causal-RL)

In [ ]:
model_path = CHECKPOINTS / "ppo_causal_rs_large_w03.zip"
if not model_path.exists():
    model_path = CHECKPOINTS / "ppo_causal_rs_w03.zip"
print(f"Loading model from: {model_path}")
try:
    model = PPO.load(str(model_path))
except Exception as e:
    print(f"Could not load model: {e}")
    model = None


In [ ]:
def replay_evaluate(policy, test_sessions, news_df, cdi_cache, w=0.6, K=10, T=10):
    results = []
    for session in test_sessions:
        env = NewsRecommendEnv([session], news_df, cdi_cache, w=w, K=20, T=T)
        obs, _ = env.reset()
        rec_lists = []
        for step in range(T):
            action, _ = policy.predict(obs, deterministic=True)
            candidates = session.candidates[step]
            candidate_embs = np.vstack([c.title_emb for c in candidates])
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0)
            with torch.no_grad():
                q_values = policy.policy.evaluate_actions(
                    obs[None], np.arange(len(candidates))
                )
            ranked = np.argsort(q_values)[::-1][:K]
            rec_lists.append([candidates[i].item_id for i in ranked])
            obs, _, done, _, _ = env.step(action)
            if done:
                break
        clicked = set(session.clicked_items)
        rec = rec_lists[-1] if rec_lists else []
        rec_embs = []
        for item_id in rec:
            row = news_df.loc[item_id]
            emb = row["I_title_emb_full"] if "I_title_emb_full" in row else np.zeros(768)
            if isinstance(emb, list):
                emb = np.array(emb, dtype=np.float32)
            rec_emds.append(emb)
        metrics = {
            "ndcg": ndcg_at_k(rec, clicked, K),
            "precision": precision_at_k(rec, clicked, K),
            "ild": ild(np.array(rec_embs)) if len(rec_embs) >= 2 else 0.0,
        }
        results.append(metrics)
    return results


news_lookup = test_df[select_cols].drop_duplicates("item_id").set_index("item_id") if "select_cols" in dir() else test_df.set_index("item_id")

ppo_results = replay_evaluate(model.policy, test_sessions, news_lookup, cdi_cache) if model else []
print(f"Evaluated {len(ppo_results)} PPO sessions")

## 5. Evaluate Baselines (Random and Popularity)

In [ ]:
random_results = []
popularity_results = []

item_popularity = test_df["item_id"].value_counts().to_dict()

for session in test_sessions:
    candidates = session.candidates[0]
    item_ids = [c.item_id for c in candidates]
    random_rec = np.random.choice(item_ids, size=min(10, len(item_ids)), replace=False).tolist()
    popular_rec = sorted(item_ids, key=lambda x: item_popularity.get(x, 0), reverse=True)[:10]
    clicked = set(session.clicked_items)
    random_embs = []
    popular_embs = []
    for item_id in random_rec:
        row = news_lookup.loc[item_id]
        emb = row["I_title_emb_full"] if "I_title_emb_full" in row else np.zeros(768)
        if isinstance(emb, list):
            emb = np.array(emb, dtype=np.float32)
        random_embs.append(emb)
    for item_id in popular_rec:
        row = news_lookup.loc[item_id]
        emb = row["I_title_emb_full"] if "I_title_emb_full" in row else np.zeros(768)
        if isinstance(emb, list):
            emb = np.array(emb, dtype=np.float32)
        popular_embs.append(emb)
    random_results.append({
        "ndcg": ndcg_at_k(random_rec, clicked, 10),
        "precision": precision_at_k(random_rec, clicked, 10),
        "ild": ild(np.array(random_embs)) if len(random_embs) >= 2 else 0.0,
    })
    popularity_results.append({
        "ndcg": ndcg_at_k(popular_rec, clicked, 10),
        "precision": precision_at_k(popular_rec, clicked, 10),
        "ild": ild(np.array(popular_embs)) if len(popular_embs) >= 2 else 0.0,
    })

print(f"Evaluated {len(random_results)} Random, {len(popularity_results)} Popularity sessions")


## 6. Aggregate Metrics and Significance Testing

In [ ]:
def aggregate_metrics(results):
    df = pd.DataFrame(results)
    return {
        "ndcg_mean": float(df["ndcg"].mean()),
        "ndcg_std": float(df["ndcg"].std()),
        "precision_mean": float(df["precision"].mean()),
        "precision_std": float(df["precision"].std()),
        "ild_mean": float(df["ild"].mean()),
        "ild_std": float(df["ild"].std()),
        "n_sessions": len(results),
    }

def significance_test(scores_a, scores_b, label_a, label_b, metric):
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)
    d = (np.mean(scores_a) - np.mean(scores_b)) / (np.std(scores_a) + 1e-10)
    sig = "SIGNIFICANT" if p_val < 0.01 else "not significant"
    print(f"{label_a} vs {label_b} ({metric}): t={t_stat:.3f}, p={p_val:.4f}, d={d:.3f} ({sig})")
    return p_val < 0.01


aggregated = {}
for name, results in [("PPO (Causal-RL)", ppo_results),
                      ("Random", random_results),
                      ("Popularity", popularity_results)]:
    if results:
        aggregated[name] = aggregate_metrics(results)
        print(f"\n{name}:")
        print(f"  NDCG@10:    {aggregated[name]['ndcg_mean']:.4f} ± {aggregated[name]['ndcg_std']:.4f}")
        print(f"  Precision@10: {aggregated[name]['precision_mean']:.4f} ± {aggregated[name]['precision_std']:.4f}")
        print(f"  ILD:        {aggregated[name]['ild_mean']:.4f} ± {aggregated[name]['ild_std']:.4f}")
        print(f"  n:          {aggregated[name]['n_sessions']}")

print("\n" + "=" * 60)
print("Significance Tests (p < 0.01):")
print("=" * 60)
if ppo_results and random_results:
    significance_test(
        [r["ndcg"] for r in ppo_results],
        [r["ndcg"] for r in random_results],
        "PPO", "Random", "NDCG"
    )
if ppo_results and popularity_results:
    significance_test(
        [r["ndcg"] for r in ppo_results],
        [r["ndcg"] for r in popularity_results],
        "PPO", "Popularity", "NDCG"
    )
